# MID training — diffusion decoder (LOO CV)

Train the main MID model (Trajectron++ encoder + DDPM diffusion decoder) with **leave-one-out cross-validation** across the 5 ETH/UCY scenes. By default we run just one fold (`SINGLE_FOLD="eth"`); set it to `None` to run all 5.

Structurally identical to `train_cvae.ipynb`, with one difference: `AutoEncoder` (diffusion) instead of `CVAEAutoEncoder` (CVAE ablation). Same encoder, same data pipeline, same eval, different decoder.

Per-fold pipeline:
1. Build the merged training Environment from the 4 non-held-out scenes (via `iter_loo_folds`).
2. Build train / val / test DataLoaders. Val = held-out scene's `_val.pkl` (used for per-epoch overfit tracking); test = held-out scene's `_test.pkl` (touched only at final eval).
3. Build a fresh encoder + `AutoEncoder` + optimizer.
4. Train. Per epoch: forward+backprop on train, then a `no_grad` val pass.
5. Evaluate Best-of-20 ADE/FDE on the held-out test split.
6. Save a per-fold checkpoint.

## 1. Configuration

In [4]:
SINGLE_FOLD = None #"eth"        # set to None to run all 5 LOO folds
BATCH_SIZE = 256
EPOCHS = 2                # paper uses 90; lower for smoke tests
LR = 1e-3
ENCODER_DIM = 256
TF_LAYER = 3
NUM_DIFFUSION_STEPS = 100
BETA_1 = 1e-4
BETA_T = 5e-2
AUGMENT = True             # picks one of the 24 precomputed rotations per sample
NUM_SAMPLES = 20           # for eval Best-of-K
SEED = 123

CHECKPOINT_DIR = "../checkpoints"
PROCESSED_DATA = "../processed_data"

## 2. Setup: paths, device, seeds

`PROJECT_ROOT` (this notebook's parent directory) needs to be on `sys.path` so `import mid_model`, `import environment`, `import models`, `import dataset`, and `import utils` all resolve to the local copies at the project root.

In [5]:
import os
import sys
import time
import random
import numpy as np
import torch

PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), ".."))
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

if torch.cuda.is_available():
    DEVICE = torch.device("cuda")
elif torch.backends.mps.is_available():
    DEVICE = torch.device("mps")
else:
    DEVICE = torch.device("cpu")
print(f"Device: {DEVICE}")

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

os.makedirs(CHECKPOINT_DIR, exist_ok=True)

Device: mps


## 3. LOO cross-validation loop

Each fold is independent: a fresh `ModelRegistrar`, a fresh `Trajectron` encoder, a fresh `AutoEncoder`, a fresh optimizer. The encoder must be rebuilt per fold because `set_environment(...)` registers one MGCVAE sub-model per node type against the *training* environment of that fold.

In [6]:
from tqdm.auto import tqdm

from mid_model import (
    build_dataloader,
    get_hyperparameters,
    load_environment,
    AutoEncoder,
    evaluate,
    iter_loo_folds,
)
from models.trajectron import Trajectron
from utils.model_registrar import ModelRegistrar

fold_results = []  # collected (held_out, ade, fde) tuples

for fold_idx, (held_out, train_env, test_env) in enumerate(
    iter_loo_folds(PROCESSED_DATA, single_fold=SINGLE_FOLD)
):
    print(f"\n{'='*60}")
    print(f"  Fold {fold_idx + 1}: held-out scene = {held_out!r}")
    print(f"  train scenes: {len(train_env.scenes)}  |  test scenes: {len(test_env.scenes)}")
    print(f"{'='*60}")

    # ---- hyperparams + dataloaders ----
    hyperparams = get_hyperparameters(encoder_dim=ENCODER_DIM)
    hyperparams["batch_size"] = BATCH_SIZE

    train_loader, node_type = build_dataloader(
        env=train_env,
        hyperparams=hyperparams,
        batch_size=BATCH_SIZE,
        shuffle=True,
        num_workers=0,
        augment=AUGMENT,
    )
    test_loader, _ = build_dataloader(
        env=test_env,
        hyperparams=hyperparams,
        batch_size=BATCH_SIZE,
        shuffle=False,
        num_workers=0,
        augment=False,
    )

    # Val split for per-epoch generalization tracking on the held-out scene.
    # We use <held_out>_val.pkl (distinct from <held_out>_test.pkl which is
    # only touched at final evaluation).
    val_env = load_environment(os.path.join(PROCESSED_DATA, f"{held_out}_val.pkl"))
    val_loader, _ = build_dataloader(
        env=val_env,
        hyperparams=hyperparams,
        batch_size=BATCH_SIZE,
        shuffle=False,
        num_workers=0,
        augment=False,
    )
    print(f"  batches/epoch: {len(train_loader)}  |  val: {len(val_loader)}  |  test: {len(test_loader)}")

    # ---- fresh encoder for this fold ----
    registrar = ModelRegistrar(model_dir=CHECKPOINT_DIR, device=DEVICE)
    encoder = Trajectron(registrar, hyperparams, DEVICE)
    encoder.set_environment(train_env)
    encoder.set_annealing_params()

    # ---- assemble diffusion model ----
    model = AutoEncoder(
        encoder=encoder,
        registrar=registrar,
        encoder_dim=ENCODER_DIM,
        num_diffusion_steps=NUM_DIFFUSION_STEPS,
        beta_1=BETA_1,
        beta_T=BETA_T,
        tf_layer=TF_LAYER,
    ).to(DEVICE)
    print(f"  params: {sum(p.numel() for p in model.parameters()):,}")

    optimizer = torch.optim.Adam(model.parameters(), lr=LR)

    # ---- smoke test ----
    batch = next(iter(train_loader))
    model.train()
    loss = model.get_loss(batch, node_type)
    assert torch.isfinite(loss), "smoke-test loss is not finite"
    print(f"  smoke-test loss: {loss.item():.4f}")

    # ---- training loop ----
    history = []        # avg train loss per epoch
    val_history = []    # avg val loss per epoch
    for epoch in range(1, EPOCHS + 1):
        # train pass
        model.train()
        epoch_losses = []
        t0 = time.time()
        pbar = tqdm(train_loader, desc=f"  [{held_out}] epoch {epoch}/{EPOCHS}", ncols=100)
        for batch in pbar:
            optimizer.zero_grad()
            loss = model.get_loss(batch, node_type)
            loss.backward()
            optimizer.step()
            epoch_losses.append(loss.item())
            pbar.set_postfix(loss=f"{loss.item():.4f}")
        train_loss = float(np.mean(epoch_losses))

        # val pass
        model.eval()
        val_losses = []
        with torch.no_grad():
            for batch in val_loader:
                val_losses.append(model.get_loss(batch, node_type).item())
        val_loss = float(np.mean(val_losses)) if val_losses else float("nan")

        elapsed = time.time() - t0
        history.append(train_loss)
        val_history.append(val_loss)
        print(f"    epoch {epoch}: train={train_loss:.4f}  val={val_loss:.4f}  ({elapsed:.1f}s)")

    # ---- evaluate on held-out test split ----
    eth_rescale = (held_out == "eth")
    results = evaluate(
        model=model,
        dataloader=test_loader,
        node_type=node_type,
        device=DEVICE,
        sample=NUM_SAMPLES,
        eth_rescale=eth_rescale,
    )
    ade = float(results["ade"])
    fde = float(results["fde"])
    print(f"  [{held_out}] Best-of-{NUM_SAMPLES}  ADE={ade:.4f}  FDE={fde:.4f}")
    fold_results.append((held_out, ade, fde))

    # ---- save checkpoint ----
    ckpt_path = os.path.join(CHECKPOINT_DIR, f"mid_loo_{held_out}.pt")
    torch.save({
        "scene": held_out,
        "epoch": EPOCHS,
        "hyperparams": hyperparams,
        "encoder_dim": ENCODER_DIM,
        "tf_layer": TF_LAYER,
        "num_diffusion_steps": NUM_DIFFUSION_STEPS,
        "registrar_state_dict": registrar.model_dict.state_dict(),
        "diffusion_state_dict": model.diffusion.state_dict(),
        "history": history,
        "val_history": val_history,
        "ade": ade,
        "fde": fde,
    }, ckpt_path)
    print(f"  saved: {ckpt_path}")


  Fold 1: held-out scene = 'eth'
  train scenes: 27  |  test scenes: 1
  batches/epoch: 469  |  val: 29  |  test: 5


/Users/dhawaldixit/projects/ddpm/mid_model/diffusion.py:168: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.self_attn.batch_first was not True(use batch_first for better inference performance)
  self.transformer_encoder = nn.TransformerEncoder(encoder_layer, num_layers=tf_layer)


  params: 7,548,644
  smoke-test loss: 1.0237


  [eth] epoch 1/2: 100%|█████████████████████████████| 469/469 [06:15<00:00,  1.25it/s, loss=0.1361]


    epoch 1: train=0.1756  val=0.1340  (387.9s)


...

  [eth] Best-of-20  ADE=0.5776  FDE=1.0293
  saved: ../checkpoints/mid_loo_eth.pt

  Fold 2: held-out scene = 'hotel'
  train scenes: 27  |  test scenes: 1
  batches/epoch: 472  |  val: 28  |  test: 10
  params: 7,548,644


/Users/dhawaldixit/projects/ddpm/mid_model/diffusion.py:168: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.self_attn.batch_first was not True(use batch_first for better inference performance)
  self.transformer_encoder = nn.TransformerEncoder(encoder_layer, num_layers=tf_layer)


  smoke-test loss: 1.0531


  [hotel] epoch 1/2:  15%|████▎                       | 72/472 [00:58<05:24,  1.23it/s, loss=0.1755]


KeyboardInterrupt: 

## 4. Summary

Per-fold ADE/FDE. If multiple folds ran, also report the mean across folds.

In [7]:
print("────────────────────────────────────────")
print("    Fold    ADE      FDE")
print("────────────────────────────────────────")
for held_out, ade, fde in fold_results:
    print(f"    {held_out:<7} {ade:<8.4f} {fde:<8.4f}")
if len(fold_results) > 1:
    mean_ade = float(np.mean([r[1] for r in fold_results]))
    mean_fde = float(np.mean([r[2] for r in fold_results]))
    print("────────────────────────────────────────")
    print(f"    {'mean':<7} {mean_ade:<8.4f} {mean_fde:<8.4f}")
print("────────────────────────────────────────")

────────────────────────────────────────
    Fold    ADE      FDE
────────────────────────────────────────
    eth     0.5776   1.0293  
────────────────────────────────────────
